# Agentic Systems

What are agents?
"Agent" can be defined in several ways. Some customers define agents as fully autonomous systems that operate independently over extended periods, using various tools to accomplish complex tasks. Others use the term to describe more prescriptive implementations that follow predefined workflows. At Anthropic, we categorize all these variations as agentic systems, but draw an important architectural distinction between workflows and agents:

- Workflows are systems where LLMs and tools are orchestrated through predefined code paths.
Agents, on the other hand, are systems where LLMs dynamically direct their own processes and tool usage, maintaining control over how they accomplish tasks.

- workflows offer predictability and consistency for well-defined tasks, whereas agents are the better option when flexibility and model-driven decision-making are needed at scale.

- CAUTION. Agentic systems often trade latency and cost for better task performance, and you should consider when this tradeoff makes sense. For many applications, however, optimizing single LLM calls with retrieval and in-context examples is usually enough.

## Contents

### Four Agentic Patterns

In this notebook, we look at four agentic patterns: (1) **reflection**, (2) **tool use**, (3) **planning** / **ReAct** [@ReAct2023], (4) **multi-agent** pattern. This notebook builds on [this blog series](https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io) by [`@deeplearning.ai`](https://www.deeplearning.ai/) and the repo by [`@neural-maze`](https://github.com/neural-maze/agentic-patterns-course?tab=readme-ov-file). These are summarized in the following table:



| Pattern              | Key Components                              | Description                                                                 |
|----------------------|---------------------------------------------|-----------------------------------------------------------|
| **Reflection**       | Generate ⟳ Reflect                          | Iterative cycle where the model produces outputs, reflects on them, and improves future generations. |
| **Tool Use**         | Tools to external systems | The model selects and applies external tools to extend its capabilities.   |
| **Planning (ReAct)** | Think ↔ Act ↔ Obs | Combines reasoning ("thought") with actions and feedback from observations in a loop. |
| **Multi-Agent**      | Agent 1 → Agent 2 → Agent 3       | Multiple agents collaborate or sequence their actions to solve complex tasks. |



## Utility functions

### Chat completion

Usual boilerplate when dealing with chat completions.

In [1]:
class Role:
    USER = "user"
    TOOL = "tool"
    SYSTEM = "system"
    ASSISTANT = "assistant"

    @classmethod
    def get_valid_roles(cls) -> set:
        """Automatically detect all uppercase constant roles"""
        return {value for name, value in vars(cls).items() 
                if name.isupper() and isinstance(value, str)}
    
    @classmethod
    def validate(cls, role: str) -> str:
        valid_roles = cls.get_valid_roles()
        if role not in valid_roles:
            raise ValueError(f"Invalid role: {role}")
        return role


def completions_create(client, messages: list, model: str) -> str:
    """Return generated string from model based on messages."""
    response = client.chat.completions.create(messages=messages, model=model)
    return str(response.choices[0].message.content)


def message_dict(prompt: str, role: str, tag: str = "") -> dict:
    """Return a message dictionary for the chat completions API."""
    role = Role.validate(role)
    prompt = f"<{tag}>{prompt}</{tag}>" if tag else prompt
    return {"role": role, "content": prompt}


# example
print(Role.get_valid_roles())

try:
    print(message_dict(role="user", prompt="Hello", tag="greeting"))
    print(message_dict(role="test", prompt="Hello", tag="greeting"))
except ValueError as e:
    print(e)

{'tool', 'system', 'assistant', 'user'}
{'role': 'user', 'content': '<greeting>Hello</greeting>'}
Invalid role: test


Implementing chat history class to abstract appending messages with limit to naively prevent ["context overflow"](https://aws.amazon.com/blogs/security/context-window-overflow-breaking-the-barrier/). We have the parameter `fixed_n` (default `1`) to preserve first `n` message since it is often important (e.g. `n=1` for the system prompt).

In [ ]:
from typing import Optional


class ChatHistory(list):
    def __init__(self, 
        system_prompt: Optional[str] = None, 
        messages: Optional[list] = None,
        max_len: int = -1, 
        fixed_n: int = 1
    ):
        """Fixed message list with a optional total length and number of fixed initial messages."""
        messages = [] if messages is None else messages
        super().__init__(messages)
        assert max_len > 1 or max_len == -1, "max_len must be -1 (no limit) or > 1"
        assert bool(system_prompt) + bool(messages) <= 1
        self.fixed_n = fixed_n
        self.max_len = max_len
        if system_prompt:
            self.update(prompt=system_prompt, role=Role.SYSTEM)
        
    def append(self, chat: dict):
        if len(self) == self.max_len:
            self.pop(self.fixed_n)    # i.e. keep 0, 1, ..., n-1 (first n)
        chat["role"] = Role.validate(chat["role"])
        super().append(chat)

    def update(self, prompt: str, role: str):
        """Append a message to the chat history."""
        self.append(message_dict(prompt=prompt, role=role))


chat_history = ChatHistory(
    system_prompt="you are a goldfish", max_len=3, fixed_n=1
)
chat_history.update("1", "user")
chat_history.update("2", "user")
chat_history.update("3", "user")
chat_history

[{'role': 'system', 'content': 'you are a goldfish'},
 {'role': 'user', 'content': '2'},
 {'role': 'user', 'content': '3'}]

In [ ]:
for cmd in [
    lambda: chat_history.append({"role": "test", "content": "test"}),
    lambda: chat_history.update(role="test", prompt="test")
]:
    try:
        cmd()
    except ValueError as e:
        print(e)

Invalid role: test
Invalid role: test


### Tag extraction

The following utilities will be used to extract content from tags (e.g. `<thought>`, `<response>`, etc).

In [ ]:
import re
from dataclasses import dataclass


@dataclass
class TagContentResult:
    content: list[str]
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """
    Extracts all content enclosed by specified tags, 
    e.g. <thought>, <response>, etc.
    Parameters:
        text (str): The input string containing multiple potential tags
        tag  (str): The name of the tag to search for
    """
    tag_pattern = rf"<{tag}>(.*?)</{tag}>"
    matched_contents = re.findall(tag_pattern, text, re.DOTALL)

    return TagContentResult(
        content=[content.strip() for content in matched_contents],
        found=bool(matched_contents),
    )

message = """
<thought>This is a thought.</thought> 
<response>This is a response.</response>
<thought>This is another thought.</thought> 
"""
print(extract_tag_content(message, "thought"))
print(extract_tag_content(message, "response"))
print(extract_tag_content(message, "tool"))

TagContentResult(content=['This is a thought.', 'This is another thought.'], found=True)
TagContentResult(content=['This is a response.'], found=True)
TagContentResult(content=[], found=False)
